# 🧬 07 — Expert v2 "População": dados REAIS + pool de oponentes + gate devastador

> **Célula única — `Runtime → Run all`.** Pré-requisitos nos **Secrets** do Colab: `GH_TOKEN` (repo privado) e `HF_TOKEN`. Use **GPU (A100, Pro+)**; pode rodar em background.

## Por que este notebook existe
A **análise devastadora** do Expert v1 descobriu: o gargalo **não é** a quantidade de jogadas
(2M de self-play) — é a **diversidade de oponentes**. Self-play puro deixa margens finas contra
estilos que ele nunca viu (NitExploiter: só +22 bb/100) e sangra ~17 bb/100 nos blinds.

## O que a célula faz (ponta a ponta)
1. **Baixa mãos REAIS** de NLHE a dinheiro — subset *handhq* (21,6M mãos, **CC BY 4.0**, [Zenodo 10796885](https://zenodo.org/records/10796885)) via espelho GitHub [uoftcprg/phh-dataset](https://github.com/uoftcprg/phh-dataset): **só a amostra necessária (~200MB)**, sem o zip de 20GB.
2. **Extrai priors populacionais** (fold-to-bet, agressão, VPIP/PFR de humanos de verdade) → `priors.json` no HF → valores iniciais pro bot **Adaptativo** (em vez do neutro 0.5).
3. **Calibra dois perfis humanos** a partir da distribuição real: **fish** (solto-passivo) e **reg** (TAG) — limiares de força calibrados empiricamente.
4. **Treina o Expert v2** (MaskablePPO, mesas paralelas) contra um **POOL diverso**: snapshot de si + random + heurístico + **sondas de exploração** (maniac/station/nit) + **fish/reg humanos** — rotacionados a cada ciclo.
5. **GATE DEVASTADOR**: bateria pareada (mesmos baralhos) v1 × v2 em 6 confrontos (random/heuristic/montecarlo/maniac/station/nit). **Só promove** para `poker_expert.onnx` no HF se o v2 vencer na soma **sem degradar nenhum confronto** além do ruído — senão sobe como candidato para inspeção.

## Depois de promovido
Baixe `poker_expert.onnx` de `felipesp1983/poker-arena-expert` e substitua
`backend/models/poker_expert.onnx`; rode a opção **[4] Validar** do launcher POKER.

*Fontes: Zenodo 10796885 (CC BY 4.0) · uoftcprg/phh-dataset · lições internas: notebook 05
(warm-start reprovado) e a auditoria devastadora (jul/2026).*


In [ ]:
# ============================================================
# EXPERT v2 "POPULACAO" - dados REAIS + pool de oponentes + gate devastador
# Celula UNICA: Run all. Pre-req (Secrets): GH_TOKEN + HF_TOKEN. GPU (A100 Pro+).
#
# O QUE ESTA CELULA FAZ (ponta a ponta):
#  1. Baixa maos REAIS de NLHE (Zenodo 10796885, CC BY 4.0 - handhq, 21.6M maos)
#  2. Extrai PRIORS populacionais (fold-to-bet, agressao, VPIP/PFR) -> Adaptativo
#  3. Calibra perfis humanos ("fish" solto-passivo, "reg" TAG) a partir dos dados
#  4. Treina o Expert v2 (PPO) contra um POOL DIVERSO: snapshot + regras + sondas
#     de exploracao (maniac/station/nit) + perfis humanos calibrados
#  5. GATE DEVASTADOR: so promove o v2 se vencer o v1 pareado, sem degradar nada
# ============================================================
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
gh = userdata.get('GH_TOKEN')
assert gh and os.environ.get('HF_TOKEN'), 'Configure GH_TOKEN e HF_TOKEN nos Secrets do Colab.'
!rm -rf poker-arena
!git clone -q https://{gh}@github.com/FELIPEACASTRO/poker-arena.git
!pip install -q poker-arena/backend 'stable-baselines3>=2.3,<3' 'sb3-contrib>=2.3,<3' 'gymnasium>=0.29,<2' onnx onnxruntime huggingface_hub requests
print('Instalado. (NAO compartilhe a saida - contem o token)')

import json
import random
import time
from collections import defaultdict

import numpy as np
import requests
import torch
import gymnasium as gym
import onnxruntime as ort
from gymnasium import spaces
from sb3_contrib import MaskablePPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import SubprocVecEnv
from huggingface_hub import HfApi

from poker_arena.bots.heuristic_bot import preflop_strength
from poker_arena.bots.ml_bot import MLBot
from poker_arena.bots.observation import Observation, observation_for
from poker_arena.engine.actions import Action, ActionType
from poker_arena.engine.cards import Deck
from poker_arena.engine.game import Hand
from poker_arena.engine.player import Player, PlayerStatus
from poker_arena.ml import FEATURE_SIZE, N_ACTIONS, SelfPlayEnv, encode, eval_bb100

try:  # tomllib e stdlib no 3.11+
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib


# ---------------- logs estilo Use a Cabeca ----------------
def box(titulo, linhas=()):
    w = max([len(titulo)] + [len(s) for s in linhas]) if linhas else len(titulo)
    print('\n  +' + '-' * (w + 2) + '+')
    print('  | ' + titulo.ljust(w) + ' |')
    print('  +' + '-' * (w + 2) + '+')
    for s in linhas:
        print('  | ' + s.ljust(w) + ' |')
    print('  +' + '-' * (w + 2) + '+')


def barra(frac, n=22):
    frac = max(0.0, min(1.0, frac))
    c = int(frac * n)
    return '[' + '#' * c + '.' * (n - c) + f'] {100 * frac:3.0f}%'


# ---------------- CONFIG ----------------
DATA_REPO = 'uoftcprg/phh-dataset'  # espelho GitHub dos dados do Zenodo 10796885
SAMPLE_HANDS = 250_000        # maos reais a parsear (de 21.6M) - de sobra p/ priors
MIN_HANDS_PER_PLAYER = 50     # so opina sobre jogador com amostra decente
N_ENVS = 8
TOTAL_STEPS = 1_500_000       # aumente com A100 em background se quiser
EVAL_EVERY = 150_000
GATE_HANDS = 400              # maos por confronto no gate (x seeds)
GATE_SEEDS = (11, 22, 33)
HF_MODEL = 'felipesp1983/poker-arena-expert'
V1_PATH = 'poker-arena/backend/models/poker_expert.onnx'

box('EXPERT v2 POPULACAO - o plano', [
    'Descoberta da analise devastadora: o gargalo NAO e quantidade de',
    'jogadas (2M de self-play) e sim DIVERSIDADE de oponentes.',
    '',
    '1. maos REAIS (Zenodo, CC BY 4.0)  -> priors do Adaptativo',
    '2. perfis humanos calibrados       -> fish (solto) e reg (TAG)',
    '3. treino vs POOL diverso          -> snapshot+regras+sondas+humanos',
    '4. gate devastador pareado         -> so promove se vencer o v1',
])

# ============================================================
# [1/5] DADOS REAIS - amostra do handhq direto do GitHub (sem zip de 20GB!)
# ============================================================
box('[1/5] Localizando maos reais (handhq - dados do Zenodo 10796885, CC BY 4.0)')
HDRS = {'Authorization': f'Bearer {gh}', 'Accept': 'application/vnd.github+json'}


def gh_ls(path):
    r = requests.get(f'https://api.github.com/repos/{DATA_REPO}/contents/{path}',
                     headers=HDRS, timeout=60)
    r.raise_for_status()
    return r.json()


folders = [f['path'] for f in gh_ls('data/handhq') if f['type'] == 'dir']
assert folders, 'nenhuma pasta em data/handhq - confira o repo do dataset'
# diversidade: 1 pasta por SITE (ABS/FTP/IPN/ONG/PS/PTY), ate 6
by_site = {}
for d in folders:
    site = d.rsplit('/', 1)[-1].split('-')[0]
    by_site.setdefault(site, d)
chosen = list(by_site.values())[:6]
print('  pastas escolhidas (1 por site, stakes variados):')
for c in chosen:
    print('   -', c.rsplit('/', 1)[-1])


def walk_phhs(path, depth=0):
    """Coleta URLs de download dos .phh/.phhs (ate 3 niveis de pasta)."""
    if depth > 3:
        return []
    out = []
    for it in gh_ls(path):
        if it['type'] == 'file' and it['name'].lower().endswith(('.phh', '.phhs')):
            out.append(it['download_url'])
        elif it['type'] == 'dir':
            out.extend(walk_phhs(it['path'], depth + 1))
    return out


phh_urls = []
for c in chosen:
    phh_urls.extend(walk_phhs(c))
random.Random(42).shuffle(phh_urls)  # mistura sites/stakes na amostra
print(f'  arquivos .phhs achados: {len(phh_urls):,} (~1.000 maos cada; '
      f'alvo: {SAMPLE_HANDS:,} maos ~ {SAMPLE_HANDS // 1000} arquivos ~200MB)')

# ============================================================
# [2/5] PRIORS populacionais (parse PHH -> stats por jogador)
# ============================================================
box('[2/5] Baixando e parseando o comportamento humano REAL')

stats = defaultdict(lambda: dict(hands=0, vpip=0, pfr=0, faced=0, folded=0, calls=0, raises=0))
parsed = skipped = 0
t0 = time.time()


def eat_hand(raw):
    """Um hand PHH -> atualiza stats (nao precisa das cartas: so acoes/apostas)."""
    players = raw.get('players') or [f'p{i + 1}' for i in range(len(raw['starting_stacks']))]
    n = len(players)
    blinds = list(raw.get('blinds_or_straddles') or [0] * n)
    cur = blinds[:n] + [0] * max(0, n - len(blinds))
    street_max = max(cur) if cur else 0
    street = 0
    vpip_set, pfr_set = set(), set()
    for tok in raw['actions']:
        parts = tok.split()
        if parts[0] == 'd':
            if parts[1] == 'db':
                street += 1
                cur = [0] * n
                street_max = 0
            continue
        seat = int(parts[0][1:]) - 1
        name = players[seat]
        verb = parts[1]
        if verb == 'sm':
            continue
        st = stats[name]
        owe = street_max - cur[seat]
        if owe > 0:
            st['faced'] += 1
        if verb == 'f':
            if owe > 0:
                st['folded'] += 1
        elif verb == 'cc':
            cur[seat] = street_max
            st['calls'] += 1
            if street == 0 and owe > 0:
                vpip_set.add(name)
        elif verb == 'cbr':
            to = float(parts[2])
            cur[seat] = to
            street_max = max(street_max, to)
            st['raises'] += 1
            if street == 0:
                vpip_set.add(name)
                pfr_set.add(name)
    for name in players:
        stats[name]['hands'] += 1
    for name in vpip_set:
        stats[name]['vpip'] += 1
    for name in pfr_set:
        stats[name]['pfr'] += 1


def hands_in(blob):
    """.phh = 1 mao por arquivo; .phhs = varias maos como secoes TOML [1],[2],..."""
    try:
        data = tomllib.loads(blob)
    except Exception:
        return []
    if 'actions' in data:
        return [data]
    return [v for v in data.values() if isinstance(v, dict) and 'actions' in v]


for url in phh_urls:
    if parsed >= SAMPLE_HANDS:
        break
    try:
        blob = requests.get(url, timeout=60).text
    except Exception:
        skipped += 1
        continue
    for raw in hands_in(blob):
        if parsed >= SAMPLE_HANDS:
            break
        try:
            # so mesas de 4+ jogadores: heads-up inflaria os priors (em HU
            # joga-se 80%+ das maos) e nosso alvo e o ring game 6-max
            if 'starting_stacks' in raw and len(raw['starting_stacks']) >= 4:
                eat_hand(raw)
                parsed += 1
                if parsed % 50_000 == 0:
                    sp = parsed / max(time.time() - t0, 1e-9)
                    print(f'  {barra(parsed / SAMPLE_HANDS)}  {sp:.0f} maos/s')
        except Exception:
            skipped += 1

pop = [s for s in stats.values() if s['hands'] >= MIN_HANDS_PER_PLAYER]
assert pop, 'amostra insuficiente - aumente SAMPLE_HANDS'


def pct(vals, q):
    vals = sorted(vals)
    return vals[min(int(q * len(vals)), len(vals) - 1)]


vpips = [s['vpip'] / s['hands'] for s in pop]
pfrs = [s['pfr'] / s['hands'] for s in pop]
f2bs = [s['folded'] / s['faced'] for s in pop if s['faced'] >= 30]
aggrs = [s['raises'] / (s['calls'] + s['raises']) for s in pop if (s['calls'] + s['raises']) >= 30]
PRIORS = {
    'source': 'handhq via uoftcprg/phh-dataset (Zenodo 10796885, CC BY 4.0), NLHE real 2009',
    'sample_hands': parsed,
    'players_measured': len(pop),
    'fold_to_bet': {'p25': pct(f2bs, .25), 'median': pct(f2bs, .5), 'p75': pct(f2bs, .75)},
    'aggression': {'p25': pct(aggrs, .25), 'median': pct(aggrs, .5), 'p75': pct(aggrs, .75)},
    'vpip': {'p25': pct(vpips, .25), 'median': pct(vpips, .5), 'p75': pct(vpips, .75)},
    'pfr': {'p25': pct(pfrs, .25), 'median': pct(pfrs, .5), 'p75': pct(pfrs, .75)},
}
json.dump(PRIORS, open('priors.json', 'w'), indent=2)
box('PRIORS POPULACIONAIS (humanos de verdade!)', [
    f"maos parseadas: {parsed:,}  (puladas: {skipped:,})  jogadores: {len(pop):,}",
    f"fold-to-bet mediano : {PRIORS['fold_to_bet']['median']:.2f}",
    f"agressao mediana    : {PRIORS['aggression']['median']:.2f}",
    f"VPIP mediano        : {PRIORS['vpip']['median']:.2f}   PFR: {PRIORS['pfr']['median']:.2f}",
    '',
    'PRO ADAPTATIVO: use estes valores como leitura INICIAL do humano',
    '(em vez do neutro 0.5) em bots/opponent_model.py - priors.json sobe pro HF.',
])

# ============================================================
# [3/5] perfis humanos calibrados (fish/reg) + sondas -> POOL
# ============================================================
box('[3/5] Calibrando perfis humanos a partir da distribuicao REAL')
# limiar de forca tal que P(preflop_strength >= t) = alvo (empirico, 20k maos)
deck_rng = random.Random(7)
samples = []
for _ in range(20_000):
    d = Deck(deck_rng.randrange(1 << 30))
    d.shuffle()
    samples.append(preflop_strength(d.deal(2)))
samples.sort()


def strength_threshold(target_frac):
    idx = max(0, min(len(samples) - 1, int((1 - target_frac) * len(samples))))
    return samples[idx]


FISH = dict(vpip=PRIORS['vpip']['p75'], pfr=PRIORS['pfr']['p25'],
            f2b=PRIORS['fold_to_bet']['p25'], aggr=PRIORS['aggression']['p25'])
REG = dict(vpip=PRIORS['vpip']['p25'], pfr=PRIORS['pfr']['p75'],
           f2b=PRIORS['fold_to_bet']['p75'], aggr=PRIORS['aggression']['p75'])
for prof in (FISH, REG):
    prof['t_vpip'] = strength_threshold(prof['vpip'])
    prof['t_pfr'] = strength_threshold(prof['pfr'])
print(f"  FISH (solto-passivo): entra {FISH['vpip']:.0%}, abre {FISH['pfr']:.0%}, folda {FISH['f2b']:.0%}")
print(f"  REG  (apertado-agr.): entra {REG['vpip']:.0%}, abre {REG['pfr']:.0%}, folda {REG['f2b']:.0%}")

# indices das features (contrato do encoder v1)
I_POT, I_TOCALL, I_STRENGTH = 104, 105, 110
I_STREET0 = 111


def spec_policy(spec, rng):
    """Constroi uma OpponentPolicy (features, mask) -> acao a partir de um spec."""

    def legal(mask):
        return [i for i, ok in enumerate(mask) if ok]

    if spec == 'random':
        return lambda f, m, r=rng: r.choice(legal(m))
    if spec == 'maniac':
        def p(f, m):
            return 4 if m[4] else (1 if m[1] else legal(m)[0])
        return p
    if spec == 'station':
        def p(f, m):
            return 1 if m[1] else legal(m)[0]
        return p
    if spec == 'nit':
        def p(f, m):
            to_call = f[I_TOCALL]
            if to_call > 0.004:  # ~mais que um min-raise em fichas normalizadas
                return 0 if m[0] else (1 if m[1] else legal(m)[0])
            if m[2]:
                return 2  # pressao minima constante
            return 1 if m[1] else legal(m)[0]
        return p
    if spec == 'heur':  # o velho heuristico, via feature de forca + pot odds
        def p(f, m):
            s, pot, tc = f[I_STRENGTH], f[I_POT], f[I_TOCALL]
            if s >= 0.72 and m[3]:
                return 3
            if tc <= 0:
                return 1 if m[1] else legal(m)[0]
            po = tc / (pot + tc) if (pot + tc) > 0 else 1.0
            if s >= po and m[1]:
                return 1
            return 0 if m[0] else legal(m)[0]
        return p
    if spec in ('fish', 'reg'):
        prof = FISH if spec == 'fish' else REG
        def p(f, m, prof=prof, r=rng):
            s, pot, tc = f[I_STRENGTH], f[I_POT], f[I_TOCALL]
            preflop = f[I_STREET0] >= 0.5
            if preflop and tc > 0:
                if s >= prof['t_pfr'] and m[3]:
                    return 3
                if s >= prof['t_vpip'] and m[1]:
                    return 1
                # abaixo da faixa: fish ainda paga as vezes (e solto), reg folda
                if spec == 'fish' and m[1] and r.random() < 0.35:
                    return 1
                return 0 if m[0] else (1 if m[1] else legal(m)[0])
            if tc <= 0:
                if s >= 0.80 and r.random() < prof['aggr'] and m[3]:
                    return 3
                return 1 if m[1] else legal(m)[0]
            po = tc / (pot + tc) if (pot + tc) > 0 else 1.0
            if r.random() < prof['f2b'] and s < po + 0.15:
                return 0 if m[0] else (1 if m[1] else legal(m)[0])
            return 1 if m[1] else legal(m)[0]
        return p
    raise ValueError(spec)


POOL = ['snapshot', 'snapshot', 'random', 'heur', 'maniac', 'station', 'nit', 'fish', 'reg']

# ============================================================
# [4/5] treino PPO contra o POOL (mesas paralelas)
# ============================================================
class GymPokerEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.env = SelfPlayEnv(n_players=6)
        self.observation_space = spaces.Box(0.0, 5.0, (FEATURE_SIZE,), dtype=np.float32)
        self.action_space = spaces.Discrete(N_ACTIONS)
        self._m = [True] * N_ACTIONS
        self._rng = random.Random()

    def reset(self, *, seed=None, options=None):
        o, self._m = self.env.reset()
        return np.asarray(o, np.float32), {}

    def step(self, a):
        o, self._m, r, d = self.env.step(int(a))
        return np.asarray(o, np.float32), float(r), bool(d), False, {}

    def action_masks(self):
        return np.asarray(self._m, bool)

    def set_opponent_spec(self, spec):
        if spec == 'snapshot':
            sess = ort.InferenceSession('current.onnx', providers=['CPUExecutionProvider'])
            iname = sess.get_inputs()[0].name
            def opp(feats, mask):
                logits = sess.run(None, {iname: np.asarray([feats], np.float32)})[0][0]
                idx = np.array([i for i, ok in enumerate(mask) if ok])
                z = logits[idx]
                z -= z.max()
                pr = np.exp(z)
                pr /= pr.sum()
                return int(np.random.choice(idx, p=pr))
            self.env.set_opponent(opp)
        else:
            self.env.set_opponent(spec_policy(spec, self._rng))


def make_env():
    return GymPokerEnv()


vec_env = SubprocVecEnv([make_env for _ in range(N_ENVS)], start_method='fork')
model = MaskablePPO('MlpPolicy', vec_env, n_steps=512, batch_size=512, ent_coef=0.01,
                    gamma=0.99, gae_lambda=0.95, policy_kwargs=dict(net_arch=[256, 256]),
                    verbose=0, device='auto')
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(HF_MODEL, repo_type='model', exist_ok=True, private=True)
api.upload_file(path_or_fileobj='priors.json', path_in_repo='priors.json',
                repo_id=HF_MODEL, repo_type='model')


def export_onnx(path):
    dev = model.device
    model.policy.to('cpu').eval()

    class _Onnx(torch.nn.Module):
        def __init__(s):
            super().__init__()
            s.fe = model.policy.features_extractor
            s.mlp = model.policy.mlp_extractor
            s.an = model.policy.action_net

        def forward(s, obs):
            f = s.fe(obs)
            lp, _ = s.mlp(f)
            return s.an(lp)

    torch.onnx.export(_Onnx(), torch.zeros(1, FEATURE_SIZE), path, input_names=['obs'],
                      output_names=['logits'],
                      dynamic_axes={'obs': {0: 'batch'}, 'logits': {0: 'batch'}},
                      opset_version=17, dynamo=False)
    model.policy.to(dev).train()


def assign_pool():
    """Distribui o pool pelas mesas (>=2 snapshots pra ancorar o self-play)."""
    specs = ['snapshot', 'snapshot'] + random.sample(POOL, k=max(0, N_ENVS - 2))
    random.shuffle(specs)
    for i in range(N_ENVS):
        vec_env.env_method('set_opponent_spec', specs[i % len(specs)], indices=[i])
    return specs


class Treino(BaseCallback):
    def __init__(self):
        super().__init__()
        self._t0 = None
        self._n = 0
        self._best = -1e9

    def _on_training_start(self):
        self._t0 = time.time()
        export_onnx('current.onnx')
        specs = assign_pool()
        box('POOL INICIAL DAS MESAS', [', '.join(specs)])
        box('NAO TEM PERGUNTA IDIOTA', [
            'P: Por que treinar contra maniac/station/nit e nao so contra mim mesmo?',
            'R: Porque a analise devastadora mostrou: self-play puro deixa margens',
            '   finas contra estilos que ele nunca viu. Poker de gente tem gente.',
        ])

    def _on_step(self):
        n = self.num_timesteps
        if n // EVAL_EVERY > self._n:
            self._n = n // EVAL_EVERY
            export_onnx('current.onnx')
            sess = ort.InferenceSession('current.onnx', providers=['CPUExecutionProvider'])
            iname = sess.get_inputs()[0].name

            def hero_act(obs):
                from poker_arena.ml.encoder import legal_mask, to_action
                logits = sess.run(None, {iname: np.asarray([encode(obs)], np.float32)})[0][0]
                mask = legal_mask(obs)
                best = max((i for i in range(N_ACTIONS) if mask[i]), key=lambda i: float(logits[i]))
                return to_action(obs, best)

            br = eval_bb100(hero_act, 'random', hands=400, seed=999)
            bh = eval_bb100(hero_act, 'heuristic', hands=400, seed=999)
            sps = n / max(time.time() - self._t0, 1e-9)
            eta = (TOTAL_STEPS - n) / sps / 60 if sps > 0 else 0
            print(f'  {barra(n / TOTAL_STEPS)}  vs random {br:+.0f}  vs heur {bh:+.0f}  '
                  f'(~{eta:.0f} min restantes)')
            if bh > self._best:
                self._best = bh
                export_onnx('best_pool.onnx')
                print(f'    ^ MARCO! melhor ate agora vs heuristico ({bh:+.0f}) - salvo')
            specs = assign_pool()
            print(f'    pool rotacionado: {", ".join(specs)}')
        return True


box('[4/5] TREINANDO o Expert v2 contra a populacao', [
    f'{N_ENVS} mesas paralelas - pool: {", ".join(sorted(set(POOL)))}',
    f'{TOTAL_STEPS:,} passos - eval a cada {EVAL_EVERY:,}',
])
model.learn(total_timesteps=TOTAL_STEPS, callback=Treino())
export_onnx('expert_v2.onnx')
print('treino concluido -> expert_v2.onnx')

# ============================================================
# [5/5] GATE DEVASTADOR - v1 vs v2, pareado, com sondas
# ============================================================
box('[5/5] GATE DEVASTADOR (a mesma regua da auditoria)')


class Maniac:
    def act(self, obs):
        if ActionType.ALL_IN in obs.legal_actions:
            return Action(ActionType.ALL_IN)
        return Action(ActionType.CALL if ActionType.CALL in obs.legal_actions else ActionType.CHECK)


class Station:
    def act(self, obs):
        if ActionType.CHECK in obs.legal_actions:
            return Action(ActionType.CHECK)
        return Action(ActionType.CALL)


class Nit:
    def act(self, obs):
        legal = obs.legal_actions
        if obs.to_call > 0:
            if obs.to_call <= 10 and ActionType.CALL in legal:
                return Action(ActionType.CALL)
            return Action(ActionType.FOLD)
        if ActionType.RAISE in legal:
            return Action(ActionType.RAISE, obs.min_raise_to)
        return Action(ActionType.CHECK)


def custom_bb100(hero, mk_villain, hands, seed, n=6, stack=1000, sb=10, bb=20):
    villains = [mk_villain(200 + i) for i in range(n - 1)]
    rng = random.Random(seed)
    total = 0
    for h in range(hands):
        players = [Player(f'P{i}', stack) for i in range(n)]
        hand = Hand(players, button=h % n, small_blind=sb, big_blind=bb,
                    seed=rng.randrange(1 << 30))
        hand.start()
        guard = 0
        while guard < 400:
            guard += 1
            while not hand.round_complete():
                seat = hand.to_act
                obs = observation_for(hand)
                a = hero.act(obs) if seat == 0 else villains[seat - 1].act(obs)
                hand.apply(a)
            live = [p for p in hand.players if p.status != PlayerStatus.FOLDED]
            if len(live) <= 1 or len(hand.board) >= 5:
                hand.resolve()
                break
            hand.advance_street()
        total += players[0].stack - stack
    return total / hands / bb * 100


from poker_arena.bots import HeuristicBot, MonteCarloBot, RandomBot

GATE = {
    'random': lambda s: RandomBot(seed=s),
    'heuristic': lambda s: HeuristicBot(seed=s),
    'montecarlo': lambda s: MonteCarloBot(seed=s, n_samples=120),
    'maniac': lambda s: Maniac(),
    'station': lambda s: Station(),
    'nit': lambda s: Nit(),
}


def battery(onnx_path):
    out = {}
    for name, mk in GATE.items():
        vals = [custom_bb100(MLBot(onnx_path, seed=s), mk, GATE_HANDS, s) for s in GATE_SEEDS]
        out[name] = sum(vals) / len(vals)
    return out


# o v1 nao vem no clone (o repo ignora *.onnx) -> baixa do HF se preciso
if not os.path.exists(V1_PATH):
    from huggingface_hub import hf_hub_download
    print('  v1 nao esta no clone - baixando poker_expert.onnx do HF...')
    V1_PATH = hf_hub_download(HF_MODEL, 'poker_expert.onnx', token=os.environ['HF_TOKEN'])

# desafiante = MELHOR checkpoint do treino (o final pode ser pior que o pico)
V2_PATH = 'best_pool.onnx' if os.path.exists('best_pool.onnx') else 'expert_v2.onnx'
print(f'  desafiante: {V2_PATH}')

print('  medindo v1 (o campeao atual)...')
v1 = battery(V1_PATH)
print('  medindo v2 (o desafiante)...')
v2 = battery(V2_PATH)

linhas = [f"{k:12} v1 {v1[k]:+7.0f}   v2 {v2[k]:+7.0f}   delta {v2[k] - v1[k]:+6.0f}" for k in GATE]
soma1, soma2 = sum(v1.values()), sum(v2.values())
linhas += ['', f"{'SOMA':12} v1 {soma1:+7.0f}   v2 {soma2:+7.0f}   delta {soma2 - soma1:+6.0f}"]
box('RESULTADO PAREADO (bb/100, mesmos baralhos)', linhas)

TOLERANCIA = 60  # bb/100 de ruido tolerado por confronto
passa = soma2 > soma1 and all(v2[k] >= v1[k] - TOLERANCIA for k in GATE) and v2['heuristic'] > 0
api.upload_file(path_or_fileobj=V2_PATH, path_in_repo='candidates/expert_v2.onnx',
                repo_id=HF_MODEL, repo_type='model')
if passa:
    api.upload_file(path_or_fileobj=V2_PATH, path_in_repo='poker_expert.onnx',
                    repo_id=HF_MODEL, repo_type='model')
    box('PROMOVIDO! o v2 e o novo campeao', [
        'subiu como poker_expert.onnx no HF. No backend local:',
        f'  baixe de {HF_MODEL} e substitua backend/models/poker_expert.onnx',
        'depois rode a opcao [4] Validar do launcher POKER.',
    ])
else:
    box('NAO PROMOVIDO (o gate segurou)', [
        'o v2 NAO superou o v1 na regua devastadora - e isso e o gate funcionando.',
        'ele subiu em candidates/ no HF pra inspecao. Ajustes possiveis:',
        '  - mais passos (TOTAL_STEPS), pool com mais fish/reg, ent_coef menor.',
    ])
print('\nFIM. priors.json + candidato(s) publicados em', HF_MODEL)
